# Walmart B2B Node Level Cost Pricing

Orchestrator notebook for the full NLC pricing pipeline.

**Steps:**
1. Set parameters & flags
2. Load data
3. Run NLC model (two-pass: min_units=8, then 4)
4. Apply pricing rules
5. Build new DSV
6. Update tests tracker
7. Save outputs
8. Upload DSV to hybris

**Optional steps** (toggle via flags):
- Inventory check (today vs yesterday cost comparison)
- Rollback handling
- National price updates
- Hybris upload

**Post-upload** (run separately, ~3 hours after hybris upload):
- FTP response validation

# Setup

In [64]:
import sys
import os
import logging

import numpy as np

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
)

from src.data.loader import DataLoader
from src.models.nlc_model import NLCModel
from src.rules.pricing_rules import PricingRulesEngine
from src.dsv.dsv_builder import DSVBuilder
from src.tracker.tracker_updater import TrackerUpdater

print(f"Project root: {project_root}")

Project root: c:\Users\valen\Desktop\WalmartPricing


# Params

In [ ]:
import pandas as pd
from src.notifications.slack_notifier import SlackNotifier

# ── Slack notifications ─────────────────────────────────────────────
slack_enabled = True                 # Toggle all Slack notifications on/off
slack_channel = "wmt-nlc"            # Slack channel to post to

# ── Core parameters ─────────────────────────────────────────────────
date_str = pd.to_datetime('today').strftime('%Y-%m-%d')
test = False
save = True
local_output = True                  # Save all outputs to local outputs/ instead of shared drive

# ── Optional steps ──────────────────────────────────────────────────
run_inventory_check = True         # Compare today vs yesterday inventory costs before pricing
apply_rollbacks = False              # Remove rollback SKUs from NLC + apply RB prices to national
update_national_prices = False      # Override national prices from Excel file
upload_to_hybris = False            # Upload DSV to hybris after saving (requires Selenium/Chrome)

# ── Margin test filter (set to None to include all) ─────────────────
margin_test_start_dates = ["2026-03-12"] #None      # e.g. 

# ── Rollbacks path (changes monthly — update each cycle) ────────────
# Drive letter is read from config/settings.yaml (shared_drive_letter).
# Only the path after the drive letter needs updating each month.
from src.adapters.module_loader import load_yaml as _load_yaml
_settings = _load_yaml("settings.yaml")
_rb_folder = _settings["shared_paths"]["rollbacks_folder"]
rollbacks_path = _rb_folder + r"\WalmartB2B Rollbacks tracker.xlsx"

# ── National price update config (only used if update_national_prices=True)
national_prices_path = None         # e.g. r"G:\...\Walmart B2B Item Report 2.20.26.xlsx"
national_prices_sheet = "National prices"
national_prices_skip_rows = 2

print(f"Date: {date_str}")
print(f"Test mode: {test}")
print(f"Test output (local): {local_output}")
print(f"Inventory check: {run_inventory_check}")
print(f"Apply rollbacks: {apply_rollbacks}")
print(f"Rollbacks file: {rollbacks_path}")
print(f"Update national prices: {update_national_prices}")
print(f"Upload to hybris: {upload_to_hybris}")

slack = SlackNotifier(channel=slack_channel, enabled=slack_enabled)
print(f"Slack notifications: {'enabled' if slack_enabled else 'disabled'} → #{slack_channel}")

# Notify pipeline start
slack.notify_pipeline_start(date_str, {
    "test": test,
    "inventory_check": run_inventory_check,
    "apply_rollbacks": apply_rollbacks,
    "update_national_prices": update_national_prices,
    "save": save,
})

Date: 2026-03-20
Test mode: False
Test output (local): True
Inventory check: True
Apply rollbacks: False
Rollbacks file: G:\Shared drives\07_Finance\07_10_Pricing_department\07_11_01_WalmartB2B\07_10_01_03 Rollbacks\WalmartB2B Rollbacks tracker.xlsx
Update national prices: False
Upload to hybris: False
Slack notifications: enabled → #wmt-nlc


In [66]:
from src.adapters.module_loader import load_yaml

if local_output:
    outputs_dir = os.path.join(project_root, "outputs")
    os.makedirs(outputs_dir, exist_ok=True)
    dsv_output_path = os.path.join(outputs_dir, f"New WalmartB2B DSV to upload {date_str}.csv")
    tracker_output_path = os.path.join(outputs_dir, f"Final node level costs tracker.csv")
    tracker_backup = False
    print(f"Test output mode: all files â†’ {outputs_dir}")
else:
    dsv_output_path = None       # DSVBuilder will use shared drive default
    tracker_output_path = None   # TrackerUpdater will use shared drive default
    tracker_backup = True
    nlc_config = load_yaml("settings.yaml")
    print(f"Production mode: files â†’ {nlc_config['shared_paths']['nlc_folder']}")

Test output mode: all files â†’ c:\Users\valen\Desktop\WalmartPricing\outputs


# [Optional] Inventory Check

Compare today's inventory costs against yesterday's to detect significant
cost shifts before running the pricing update. Toggle via `run_inventory_check`
in the Params cell.

Produces:
- **Summary**: count of SKU-Warehouse pairs by delta category (Increase / Decrease / No change)
- **Vendor breakdown**: which vendors had the most price increases, filtered to vendors with 1000+ lines

In [ ]:
if run_inventory_check:
    from src.data.inventory_checker import InventoryChecker

    checker = InventoryChecker(date_current=date_str)
    inv_check = checker.run()

    print(f"=== Inventory Cost Comparison: {date_str} vs {inv_check['date_previous']} ===")
    display(inv_check["df_summary"])

    print(f"\n=== Brand Increases (top 5, 1000+ lines) ===")
    display(inv_check["df_brand_increases"].head(5))

    print(f"\n=== Brand Decreases (top 5, 1000+ lines) ===")
    display(inv_check["df_brand_decreases"].head(5))

    print(f"\n=== Vendor Increases (top 5, 1000+ lines) ===")
    display(inv_check["df_vendor_increases"].head(5))

    print(f"\n=== Vendor Decreases (top 5, 1000+ lines) ===")
    display(inv_check["df_vendor_decreases"].head(5))

    slack.notify_inventory_check(inv_check, date_str)
else:
    print("[Skipped] Inventory check")
    slack.notify_inventory_check_skipped()

# Step 1-2: Load data & run NLC model

In [68]:
loader = DataLoader()
model = NLCModel(date_str=date_str)
model.load_data(loader, rollbacks_path=rollbacks_path)
df_output = model.run()

print(f"NLC output: {len(df_output)} SKU-Node rows")
display(df_output["Final target"].value_counts())
display(df_output["current_nlc_margin category"].value_counts())

slack.notify_nlc_model(len(df_output))

2026-03-20 11:22:01,971 [INFO] src.models.nlc_model: Loading NLC model data for date=2026-03-20
2026-03-20 11:22:01,978 [INFO] src.adapters.google_api_adapter: Initializing Google API service connection...
2026-03-20 11:22:03,372 [INFO] googleapiclient.discovery_cache: file_cache is only supported with oauth2client<4.0.0
2026-03-20 11:22:03,394 [INFO] googleapiclient.discovery_cache: file_cache is only supported with oauth2client<4.0.0
2026-03-20 11:22:03,399 [INFO] src.adapters.google_api_adapter: Google API service connected.
2026-03-20 11:22:06,942 [INFO] src.data.loader: Max DSV date: 2026-03-19 00:00:00
2026-03-20 11:22:06,943 [INFO] src.data.loader: Reading DSV file: dsvpriceupload-walmartb2b-2026-03-19_07-32-12.csv


Download 88.
Download 100.


2026-03-20 11:22:14,372 [INFO] src.data.loader: Loading source: warehouse_node_mapping (type=folder)


Last modified time 2026-03-20T10:00:51.272Z
Download 100.


2026-03-20 11:22:16,548 [INFO] src.data.loader: Loading source: dw_walmart_item_report (type=sql)
2026-03-20 11:22:16,549 [INFO] src.adapters.dw_adapter: Data warehouse module loaded.
2026-03-20 11:22:42,836 [INFO] src.data.loader: Loading source: dw_map_prices (type=sql)
2026-03-20 11:22:46,786 [INFO] src.data.loader: Loading source: shipping_costs_by_node (type=local)
2026-03-20 11:22:46,876 [INFO] src.data.loader: Loading source: rollbacks (type=local)
2026-03-20 11:22:47,084 [INFO] src.models.nlc_model: Rollbacks loaded: 1719 active (from 2080 total)
2026-03-20 11:22:47,085 [INFO] src.data.loader: Loading source: dw_walmart_sales (type=sql)
2026-03-20 11:23:33,862 [INFO] src.models.nlc_model: Total SKU-Node revenue (last 90 days): 89684817


...

Download 100.
Last modified time 2026-03-20T10:01:57.331Z
Download 95.
Download 100.


2026-03-20 11:26:44,295 [INFO] src.models.nlc_model: Inventory loaded for 2026-03-20: 2198298 rows
2026-03-20 11:26:53,551 [INFO] src.models.nlc_model: Total inventory rows (excl rollbacks): 15078075
2026-03-20 11:26:53,653 [INFO] src.data.loader: Loading source: tests_tracker (type=local)
c:\Users\valen\Desktop\WalmartPricing\src\data\loader.py:150: DtypeWarning: Columns (0: Final price category, 1: Final price change category final, 2: Category inventory, 3: Sub-group, 4: Last price update) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, **kwargs)
2026-03-20 11:27:19,925 [INFO] src.models.nlc_model: All NLC model data loaded.
2026-03-20 11:27:19,975 [INFO] src.models.nlc_model: Running NLC model...
2026-03-20 11:29:52,307 [INFO] src.models.nlc_model: NLC calc (min_units=8): 1563310 total, 1487295 workable
2026-03-20 11:29:59,032 [INFO] src.models.nlc_model: Pass 1 (min_units=8): 1487295 workable rows
2026-03-20 11:34:14,489 [INFO] 

NLC output: 2242963 SKU-Node rows


Final target
Added                       708079
Margin test                 362130
Normal strategy             321695
Updated                     284748
No sales test               201391
Decreased - margin > 20%    126628
Updates test                101826
Wm margin split test         88605
Shipping cost added          24423
Less sales                    2582
Less sales back               2510
Name: count, dtype: int64

current_nlc_margin category
10.8% to <11.2%    704135
11.2% to <13%      365138
8% to <10.8%       284903
13% to <15%        269122
15% to <17%        158541
6% to <8%          158255
17% to <20%        133013
>=20%              111095
0% to <6%           39136
<0%                  1388
Name: count, dtype: int64

## New SKU-Nodes + checks

In [69]:
df_new = df_output[df_output["New NLC"] == "Yes"].copy()
print(f"New SKU-Nodes: {len(df_new)}")
display(df_new["Min units"].value_counts())
display(df_new["Brand code"].value_counts().head(10))
display(df_new["Identifier"].value_counts().head(10))

New SKU-Nodes: 18237


Min units
8    11215
4     7022
Name: count, dtype: int64

Brand code
PIRE    2080
CONT    1677
KUMH    1576
NEXE    1447
FALK    1230
HANK    1183
GENE    1076
GOYR     893
NOKI     708
CARL     651
Name: count, dtype: int64

Identifier
628326175    2635
62832692     2383
628326176    1950
628326172    1843
62832691     1583
628326896     781
628326902     460
628326724     267
62832673      206
628326179     159
Name: count, dtype: int64

## Update current NLC margins & stock status

In [70]:
df_current_tests = model.df_current_tests.copy()
print(f"Tracker entries: {len(df_current_tests)}")
display(df_current_tests["Final target"].value_counts())

Tracker entries: 3287363


Final target
Added                       1266186
Normal strategy              496010
Margin test                  430867
Updated                      363179
No sales test                290282
Updates test                 147260
Decreased - margin > 20%     134388
Wm margin split test          95983
Shipping cost added           31921
Less sales                    15796
Less sales back               15491
Name: count, dtype: int64

# Step 3: Pricing rules

In [71]:
engine = PricingRulesEngine(
    df_output, model.df_current_tests, date_str, test_mode=test
)

## Walmart margin split test update

In [72]:
df_wm_split_dsv, df_wm_split_tracker = engine.get_wm_margin_split_updates()
print(f"Wm margin split DSV updates: {len(df_wm_split_dsv)}")
if len(df_wm_split_dsv) > 0:
    display(df_wm_split_dsv.head())

2026-03-20 11:35:36,240 [INFO] src.rules.pricing_rules: Wm margin split updates: 2496 SKU-Nodes


Wm margin split DSV updates: 2496


,SKU,Source,Price,SKU-Node
26176,ARRO0071416565H,628326466,38.84,ARRO0071416565H-628326466
26724,ARRO0071721565H,628326466,70.47,ARRO0071721565H-628326466
29392,ATLA0061721555W2,628326870,58.09,ATLA0061721555W2-628326870
29649,ATLA0061927555W2,628326870,112.25,ATLA0061927555W2-628326870
30386,ATLA0271620570V,628326984,63.33,ATLA0271620570V-628326984


## Brands margin test update

In [73]:
df_margin_dsv, df_margin_tracker = engine.get_margin_test_updates(
    start_dates=margin_test_start_dates
)
print(f"Margin test DSV updates: {len(df_margin_dsv)}")
if len(df_margin_dsv) > 0:
    display(df_margin_dsv.head())

2026-03-20 11:35:45,250 [INFO] src.rules.pricing_rules: Margin test updates: 1811 SKU-Nodes


Margin test DSV updates: 1811


,SKU,Source,Price,SKU-Node
907,ACCE0321924555Y,628326855,92.07,ACCE0321924555Y-628326855
929,ACCE0321925555VXL,6283261264,92.20,ACCE0321925555VXL-6283261264
1274,ACCE0322126540YXL,628326984,114.55,ACCE0322126540YXL-628326984
1374,ACCE0322129535Y,628326984,120.79,ACCE0322129535Y-628326984
2069,ACCE0401420560H,628326855,53.83,ACCE0401420560H-628326855


## Low price updates

In [74]:
df_low_dsv, df_low_tracker = engine.get_low_price_updates()
print(f"Low price DSV updates: {len(df_low_dsv)}")
if len(df_low_dsv) > 0:
    display(df_low_tracker["Category inventory"].value_counts())
    display(df_low_tracker["Min units"].value_counts())

2026-03-20 11:35:50,555 [INFO] src.rules.pricing_rules: Low price updates: 3149 SKU-Nodes


Low price DSV updates: 3149


Category inventory
Not showing inventory    1902
Below 6% margin          1247
Name: count, dtype: int64

Min units
8    1789
4    1360
Name: count, dtype: int64

## High price updates

In [75]:
df_high_dsv, df_high_tracker = engine.get_high_price_updates()
print(f"High price DSV updates: {len(df_high_dsv)}")
if len(df_high_tracker) > 0:
    summary = (
        df_high_tracker.groupby("Final target")
        .agg(
            count=("Final target", "count"),
            avg_delta=("Final delta vs current %", "mean"),
        )
        .reset_index()
    )
    display(summary)

2026-03-20 11:35:51,293 [INFO] src.rules.pricing_rules: High price updates: 1893 SKU-Nodes


High price DSV updates: 1893


,Final target,count,avg_delta
0,Decreased - margin > 20%,1893,-0.054974


## New SKU-Nodes

In [76]:
df_new_nodes_dsv, df_new_nodes_tracker = engine.get_new_sku_nodes()
print(f"New SKU-Nodes for DSV: {len(df_new_nodes_dsv)}")
if len(df_new_nodes_tracker) > 0:
    display(df_new_nodes_tracker["Final price category"].value_counts())

slack.notify_pricing_rules({
    "Wm margin split": len(df_wm_split_dsv),
    "Margin test": len(df_margin_dsv),
    "Low price updates": len(df_low_dsv),
    "High price updates": len(df_high_dsv),
    "New SKU-Nodes": len(df_new_nodes_dsv),
})

2026-03-20 11:35:51,868 [INFO] src.rules.pricing_rules: New SKU-Nodes: 18237


New SKU-Nodes for DSV: 18237


Final price category
11%    17676
8%       383
6%       178
Name: count, dtype: int64

# Step 4: Build new DSV

In [77]:
builder = DSVBuilder(
    df_curr_dsv_original=model.df_curr_dsv_original,
    today_str=date_str,
)

# Start from current DSV
df_dsv_start = builder._prepare_starting_dsv()
print(f"Starting DSV: {len(df_dsv_start)} rows")

Starting DSV: 3228730 rows


## [Optional] Update national prices

Toggle: `update_national_prices` flag in Params cell.

In [78]:
if update_national_prices:
    print(f"Updating national prices from: {national_prices_path}")
    df_dsv_start = builder.apply_national_price_updates(
        df_dsv_start,
        national_prices_path=national_prices_path,
        sheet_name=national_prices_sheet,
        skip_rows=national_prices_skip_rows,
    )
else:
    print("[Skipped] Update national prices")
slack.notify_national_prices(applied=update_national_prices)

[Skipped] Update national prices


## [Optional] Rollback handling

Toggle: `apply_rollbacks` flag in Params cell.

When enabled:
- Removes rollback SKUs from NLC rows
- Applies rollback prices to national rows

In [79]:
if apply_rollbacks:
    print(f"Active rollbacks: {len(model.df_rollbacks)} rows, "
          f"{model.df_rollbacks['Product Code'].nunique()} unique SKUs")
    df_dsv_start = builder.apply_rollbacks(df_dsv_start, model.df_rollbacks)
    slack.notify_rollbacks(
        applied=True,
        n_rollbacks=len(model.df_rollbacks),
        n_skus=model.df_rollbacks["Product Code"].nunique(),
    )
else:
    print("[Skipped] Rollback handling")
    slack.notify_rollbacks(applied=False)

[Skipped] Rollback handling


## Apply NLC updates & build final DSV

In [80]:
list_dsv_updates = [df_wm_split_dsv, df_margin_dsv, df_low_dsv, df_high_dsv]

df_new_dsv = builder.build_from(
    df_dsv_start,
    list_dsv_updates=list_dsv_updates,
    df_new_nodes=df_new_nodes_dsv,
)

print(f"Final DSV: {len(df_new_dsv)} rows")
df_new_dsv.head()

2026-03-20 11:36:02,678 [INFO] src.dsv.dsv_builder: Starting DSV: 3228730 rows
2026-03-20 11:36:02,699 [INFO] src.dsv.dsv_builder: Original: 3228730 | Updates: 9349 | New nodes: 18237 | Expected final: 3246967
2026-03-20 11:36:09,544 [INFO] src.dsv.dsv_builder: New DSV built: 3246967 rows


Final DSV: 3246967 rows


,SKU,Price,Minimum margin,Source
0,IRON0061417565H,40.11,4%,62832635
1,IRON0081417565H,40.11,4%,628326503
2,IRON0061417565H,40.11,4%,628326106
3,IRON0061417565H,40.11,4%,6283261141
4,APLS0031518555V,40.11,4%,628326916


## DSV validation

In [81]:
df_validation = builder.validate(df_new_dsv)

display(df_validation["Price change category"].value_counts())

df_changes = df_validation[df_validation["Price change category"] != "No change"]
df_changes["Is national?"] = np.where(
    df_changes["Source"] == "National", "Yes", "No"
)
display(df_changes["Is national?"].value_counts())
df_changes.head(10)

slack.notify_dsv_build(
    len(df_new_dsv),
    df_validation["Price change category"].value_counts().to_dict(),
)

2026-03-20 11:36:24,375 [INFO] src.dsv.dsv_builder: DSV validation — changes: 27565
2026-03-20 11:36:25,556 [INFO] src.dsv.dsv_builder:   {'No change': 3219402, 'New': 18237, 'Increase': 4721, 'Decrease': 4607}


Price change category
No change    3219402
New            18237
Increase        4721
Decrease        4607
Name: count, dtype: int64

Is national?
No    27565
Name: count, dtype: int64

# Step 5: Save DSV

In [ ]:
if save:
    dsv_path = builder.save(df_new_dsv, output_path=dsv_output_path)
    print(f"DSV saved to: {dsv_path}")

    # Record last run date for inventory check comparison
    from src.data.inventory_checker import save_last_run_date
    save_last_run_date(date_str)
    print(f"Last run date saved: {date_str}")
else:
    dsv_path = None
    print("[Skipped] DSV save (save=False)")

# Step 6: Update tracker

In [83]:
updater = TrackerUpdater(model.df_current_tests, today_str=date_str)

# Refresh margins from model output
updater.update_margins(df_output)

# Append all new/updated entries
updater.append_entries([
    df_new_nodes_tracker,
    df_low_tracker,
    df_high_tracker,
    df_wm_split_tracker,
    df_margin_tracker,
])

print(f"Tracker: {len(updater.df_tracker)} total rows")
print(f"Expected: {len(model.df_current_tests) + len(df_new_nodes_tracker)}")

display(updater.df_tracker["Final target"].value_counts().sort_index())

slack.notify_tracker_update(len(updater.df_tracker))

2026-03-20 11:37:00,981 [INFO] src.tracker.tracker_updater: Tracker margins updated.
2026-03-20 11:37:07,918 [INFO] src.tracker.tracker_updater: Tracker updated: 3305533 total rows


Tracker: 3305533 total rows
Expected: 3305600


Final target
Added                       1282644
Decreased - margin > 20%     135949
Less sales                    15765
Less sales back               15459
Margin test                  430862
No sales test                289804
Normal strategy              495207
Shipping cost added           31921
Updated                      364996
Updates test                 146943
Wm margin split test          95983
Name: count, dtype: int64

In [84]:
if save:
    tracker_path = updater.save(output_path=tracker_output_path, backup=tracker_backup)
    print(f"Tracker saved to: {tracker_path}")
    slack.notify_save(dsv_path=dsv_path, tracker_path=tracker_path)
else:
    tracker_path = None
    print("[Skipped] Tracker save (save=False)")
    slack.notify_save(skipped=True)

2026-03-20 11:37:52,193 [INFO] src.tracker.tracker_updater: Tracker saved: c:\Users\valen\Desktop\WalmartPricing\outputs\Final node level costs tracker.csv (3305533 rows)


Tracker saved to: c:\Users\valen\Desktop\WalmartPricing\outputs\Final node level costs tracker.csv


# Step 7: [Optional] Upload DSV to hybris

Toggle: `upload_to_hybris` flag in Params cell.

Opens a Chrome browser, signs into hybris backoffice, navigates to the
DSV Prices page, selects the WalmartB2B channel, uploads the CSV file,
and waits for the upload to finish (Status=FINISHED, Result=SUCCESS).

Requires: Selenium, Chrome, `webdriver_manager`, and hybris credentials
on the shared drive.

In [ ]:
if upload_to_hybris:
    if not save:
        print("ERROR: save=False but upload_to_hybris=True. Save the DSV first.")
    else:
        from src.dsv.hybris_uploader import HybrisUploader, copy_dsv_to_archive

        print(f"Uploading DSV to hybris: {dsv_path}")
        with HybrisUploader(headless=False) as uploader:
            success = uploader.upload(dsv_path, wait_for_result=True)

        if success:
            print("Hybris upload successful.")
            print("FTP response validation can be run in ~3 hours.")

            # Copy DSV to shared drive archive
            try:
                archive_path = copy_dsv_to_archive(dsv_path)
                print(f"DSV archived to: {archive_path}")
                slack.notify_dsv_archive(dest_path=archive_path)
            except Exception as e:
                print(f"WARNING: Failed to archive DSV: {e}")
                slack.notify_dsv_archive(error=str(e))
        else:
            print("Hybris upload failed or timed out. Check hybris manually.")
        slack.notify_hybris_upload(success=success)
else:
    print("[Skipped] Hybris upload")
    slack.notify_hybris_upload(skipped=True)

# Cleanup

In [86]:
loader.close()

slack.notify_pipeline_complete({
    "nlc_rows": len(df_output),
    "dsv_rows": len(df_new_dsv),
    "wm_split": len(df_wm_split_dsv),
    "margin_test": len(df_margin_dsv),
    "low_price": len(df_low_dsv),
    "high_price": len(df_high_dsv),
    "new_nodes": len(df_new_nodes_dsv),
    "tracker_rows": len(updater.df_tracker),
    "dsv_path": dsv_path,
    "tracker_path": tracker_path,
})
print("Done.")

2026-03-20 11:37:53,785 [INFO] src.adapters.google_api_adapter: Google API service connection closed.
2026-03-20 11:37:53,798 [INFO] src.adapters.dw_adapter: Data warehouse adapter closed.


Done.


---

# Post-upload: FTP validation

Run this section **~3 hours after uploading** the DSV via hybris.

Downloads XML response files from Walmart FTP, parses ingestion status,
and generates a summary report. Alerts if failure rate exceeds 1.5%.

In [87]:
from src.dsv.ftp_validator import FTPValidator

validator = FTPValidator(today_str=date_str)
n_files = validator.download_responses()
print(f"Downloaded {n_files} response files")

Connected to FTP server: upload.tires-easy.com


2026-03-20 11:37:58,279 [WARNING] src.dsv.ftp_validator: No response files found for 2026-03-20


Downloaded 0 response files


In [88]:
if n_files > 0:
    df_results = validator.parse_responses()
    display(df_results["ingestionStatus"].value_counts())

    report_path = validator.generate_report(df_results)
    print(f"Report saved: {report_path}")
    slack.notify_ftp_validation(n_files, df_results, report_path)
else:
    print("No response files yet. Try again later.")
    slack.notify_ftp_validation(0)

No response files yet. Try again later.
